In [ ]:
import pandas as pd
import io
from google.colab import files

# 1. Subir el archivo .xlsx
print("Sube tu archivo EXCEL (.xlsx) original:")
uploaded = files.upload()

# 2. Leer el archivo
nombre_archivo = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[nombre_archivo]))

print(f"\nProcesando '{nombre_archivo}'...")

# 3. Preparación de columnas (Desde la columna 2 en adelante)
cols_categoricas = [df.columns[1], df.columns[2]]
cols_yes_no = df.columns[3:]

# Transformación a 1s y 0s
atributos_yes = df[cols_yes_no].applymap(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)
atributos_cat = pd.get_dummies(df[cols_categoricas]).astype(int)
atributos = pd.concat([atributos_cat, atributos_yes], axis=1)

n_total = len(df)
todos_los_resultados = []

# 4. Cálculo de TODAS las combinaciones posibles (sin filtrar aún)
cols = atributos.columns
for i in range(len(cols)):
    for j in range(len(cols)):
        if i == j: continue

        A = cols[i]
        B = cols[j]

        count_A = atributos[A].sum()
        count_B = atributos[B].sum()
        count_A_and_B = ((atributos[A] == 1) & (atributos[B] == 1)).sum()

        # Fórmulas
        soporte = count_A_and_B / n_total

        if soporte > 0: # Guardamos todo lo que tenga al menos una coincidencia
            confianza = count_A_and_B / count_A if count_A > 0 else 0
            prob_B = count_B / n_total
            lift = confianza / prob_B if prob_B > 0 else 0

            todos_los_resultados.append({
                'Antecedente (A)': A,
                'Consecuente (B)': B,
                'Soporte': round(soporte, 4),
                'Confianza': round(confianza, 4),
                'Lift': round(lift, 4)
            })

# 5. Crear el DataFrame con el total de reglas
df_total = pd.DataFrame(todos_los_resultados)

# 6. FILTRAR SOLO PARA MOSTRAR EN PANTALLA (Support > 0.2)
df_pantalla = df_total[df_total['Soporte'] > 0.2].sort_values(by='Soporte', ascending=False)

print(f"\n--- REGLAS CON SOPORTE > 0.2 (Total: {len(df_pantalla)}) ---")
display(df_pantalla)

# 7. GENERAR EXCEL CON TODAS LAS REGLAS (Sin el filtro de 0.2)
nombre_salida = 'Todas_las_Reglas_Apriori.xlsx'
df_total.to_excel(nombre_salida, index=False)

print(f"\n--- GENERANDO EXCEL COMPLETO ---")
print(f"Se han guardado {len(df_total)} reglas en total.")
files.download(nombre_salida)